# 14.07 - Image Baseline Experiment

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** Image baseline from scratch.

Today is a consolidation day: under time pressure, build a complete image-classification baseline from a blank notebook. The goal is not a fancy model. The goal is a reliable contest-shaped loop: data table, label mapping, dataset, model, validation metric, and submission file.

## Core Ideas

A strong baseline is a small system with clear contracts:

- The file table tells you which image belongs to which split and label.
- The label map is created once and reused for training, validation, inference, and submission.
- The dataset returns tensors with shape `[C, H, W]`, `float32` image values, and `long` class labels.
- The model returns logits with shape `[batch, num_classes]`.
- Validation reports both loss and a competition-friendly metric such as Macro-F1.
- The submission writer converts predicted indices back into original label strings.

This notebook uses a tiny local shape dataset so the full experiment can run without downloads. Treat it like a contest rehearsal: keep each piece simple, inspect shapes constantly, and make the handoff from validation to submission explicit.

In [1]:
import os
import csv
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import torch
from torch import nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = "_day14_image_data"
IMAGE_DIR = os.path.join(DATA_DIR, "images")
LABELS_CSV = os.path.join(DATA_DIR, "labels.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
SUBMISSION_CSV = os.path.join(DATA_DIR, "submission.csv")
LABELS = ["circle", "square", "triangle"]
IMAGE_SIZE = 48
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Prepared Image Data

The provided cell creates a deterministic three-class image dataset under `_day14_image_data/`.

It writes:

- `images/*.png`
- `labels.csv` with `image_id`, `image_path`, `label`, and `split`
- `test.csv` with test image IDs and paths

Run this cell before the exercises. The generator is complete; the baseline pipeline starts after the data exists.

In [2]:
def _draw_shape_image(label, seed, image_size=IMAGE_SIZE):
    rng = np.random.default_rng(seed)
    background = np.zeros((image_size, image_size, 3), dtype=np.uint8)
    background[:, :, :] = rng.integers(12, 32, size=3)

    image = Image.fromarray(background)
    draw = ImageDraw.Draw(image)

    base_margin = 9
    shift_x = int(rng.integers(-3, 4))
    shift_y = int(rng.integers(-3, 4))
    x0 = base_margin + shift_x
    y0 = base_margin + shift_y
    x1 = image_size - base_margin + shift_x
    y1 = image_size - base_margin + shift_y

    colors = {
        "circle": (238, 77, 89),
        "square": (68, 205, 124),
        "triangle": (78, 142, 238),
    }
    fill = colors[label]

    if label == "circle":
        draw.ellipse([x0, y0, x1, y1], fill=fill)
    elif label == "square":
        draw.rectangle([x0, y0, x1, y1], fill=fill)
    else:
        points = [
            (image_size // 2 + shift_x, base_margin + shift_y),
            (image_size - base_margin + shift_x, image_size - base_margin + shift_y),
            (base_margin + shift_x, image_size - base_margin + shift_y),
        ]
        draw.polygon(points, fill=fill)

    array = np.asarray(image).astype(np.int16)
    noise = rng.normal(0, 4, size=array.shape).astype(np.int16)
    array = np.clip(array + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(array)


def make_day14_image_data(
    data_dir=DATA_DIR,
    labels=LABELS,
    image_size=IMAGE_SIZE,
    n_labeled_per_class=16,
    n_val_per_class=4,
    n_test_per_class=2,
):
    image_dir = os.path.join(data_dir, "images")
    os.makedirs(image_dir, exist_ok=True)

    labeled_rows = []
    test_rows = []

    for label_idx, label in enumerate(labels):
        for item_idx in range(n_labeled_per_class):
            split = "val" if item_idx >= n_labeled_per_class - n_val_per_class else "train"
            image_id = "%s_%s_%02d" % (split, label, item_idx)
            file_name = image_id + ".png"
            rel_path = os.path.join("images", file_name)
            seed = 1000 + label_idx * 100 + item_idx
            _draw_shape_image(label, seed, image_size).save(os.path.join(data_dir, rel_path))
            labeled_rows.append({
                "image_id": image_id,
                "image_path": rel_path,
                "label": label,
                "split": split,
            })

        for item_idx in range(n_test_per_class):
            image_id = "test_%s_%02d" % (label, item_idx)
            file_name = image_id + ".png"
            rel_path = os.path.join("images", file_name)
            seed = 5000 + label_idx * 100 + item_idx
            _draw_shape_image(label, seed, image_size).save(os.path.join(data_dir, rel_path))
            test_rows.append({
                "image_id": image_id,
                "image_path": rel_path,
                "expected_label": label,
            })

    labels_csv = os.path.join(data_dir, "labels.csv")
    test_csv = os.path.join(data_dir, "test.csv")

    with open(labels_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["image_id", "image_path", "label", "split"])
        writer.writeheader()
        writer.writerows(labeled_rows)

    with open(test_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["image_id", "image_path", "expected_label"])
        writer.writeheader()
        writer.writerows(test_rows)

    return pd.DataFrame(labeled_rows), pd.DataFrame(test_rows)


labels_df, test_df = make_day14_image_data()
print(labels_df["split"].value_counts().to_dict())
labels_df.head()


{'train': 36, 'val': 12}


,image_id,image_path,label,split
0,train_circle_00,images\train_circle_00.png,circle,train
1,train_circle_01,images\train_circle_01.png,circle,train
2,train_circle_02,images\train_circle_02.png,circle,train
3,train_circle_03,images\train_circle_03.png,circle,train
4,train_circle_04,images\train_circle_04.png,circle,train


## Exercise 14-A: Records and Label Mapping

Write `load_baseline_records(labels_csv, test_csv)`.

Return five objects:

1. `train_records`
2. `val_records`
3. `test_records`
4. `label_to_idx`
5. `idx_to_label`

Use the `split` column for train and validation records. Build the label mapping from the labeled data only, and keep it stable by sorting label names.

In [3]:
# TODO 14-A
def load_baseline_records(labels_csv=LABELS_CSV, test_csv=TEST_CSV):
    labels_df = pd.read_csv(labels_csv)
    test_df = pd.read_csv(test_csv)
    labels = sorted(labels_df["label"].unique().tolist())
    label_to_idx = {label : idx for idx, label in enumerate(labels)}
    idx_to_label = {val : key for key, val in label_to_idx.items()}
    train_records = labels_df[labels_df["split"] == "train"].reset_index(drop = True)
    val_records = labels_df[labels_df["split"] == "val"].reset_index(drop = True)
    test_records = test_df
    return train_records, val_records, test_records, label_to_idx, idx_to_label

# Smoke check: run this after implementing the functions above.
train_records, val_records, test_records, label_to_idx, idx_to_label = load_baseline_records()
print(
    "record smoke check:",
    len(train_records), len(val_records), len(test_records), label_to_idx,
)


record smoke check: 36 12 6 {'circle': 0, 'square': 1, 'triangle': 2}


## Exercise 14-B: Dataset and DataLoaders

Write the image input pipeline.

`image_to_tensor(image)` should convert a PIL image into a normalized `torch.float32` tensor with shape `[3, IMAGE_SIZE, IMAGE_SIZE]`.

`BaselineImageDataset` should return dictionaries. Labeled examples need `image`, `label`, and `image_id`; test examples need `image` and `image_id`.

`make_baseline_loaders(...)` should return train, validation, and test `DataLoader` objects. Shuffle only the training loader.

In [4]:
# TODO 14-B
def image_to_tensor(image):
    image = np.array(image)
    image = torch.tensor(image,dtype = torch.float32)
    image = image.permute(2,0,1)
    image /= 255.0
    return image


class BaselineImageDataset(Dataset):
    def __init__(self, records, data_dir=DATA_DIR, label_to_idx=None, is_test=False):
        self.records = records
        self.data_dir = data_dir
        self.label_to_idx = label_to_idx
        self.is_test = is_test

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records.iloc[idx]
        path = os.path.join(self.data_dir,row["image_path"])
        image = Image.open(path).convert("RGB")
        image = image_to_tensor(image)
        image_id = row["image_id"]
        if self.is_test : 
            return {
                "image" : image,
                "image_id" : image_id
            }
        label = self.label_to_idx[row["label"]]
        return {
            "image" : image,
            "label" : label,
            "image_id" : image_id
        }



def make_baseline_loaders(train_records, val_records, test_records, label_to_idx, batch_size=8):
    train_ds = BaselineImageDataset(train_records, label_to_idx = label_to_idx)
    val_ds = BaselineImageDataset(val_records, label_to_idx = label_to_idx)
    test_ds = BaselineImageDataset(test_records, label_to_idx = label_to_idx, is_test = True)
    train_loader = DataLoader(train_ds, batch_size = batch_size, shuffle = True)
    val_loader = DataLoader(val_ds, batch_size = batch_size, shuffle = False)
    test_loader = DataLoader(test_ds, batch_size = batch_size, shuffle = False)
    return train_loader, val_loader, test_loader


# Smoke check: run this after implementing the functions above.
train_loader, val_loader, test_loader = make_baseline_loaders(
    train_records, val_records, test_records, label_to_idx
)
smoke_batch = next(iter(train_loader))
print(
    "data smoke check:",
    smoke_batch["image"].shape,
    smoke_batch["image"].dtype,
    smoke_batch["label"].shape,
)


data smoke check: torch.Size([8, 3, 48, 48]) torch.float32 torch.Size([8])


## Exercise 14-C: CNN Baseline Model

Write a small CNN that accepts `[B, 3, 48, 48]` tensors and returns logits with shape `[B, num_classes]`.

Use this same mental pattern for transfer learning later: a feature extractor produces a compact feature vector, then a classifier head maps features to class logits.

In [5]:
# TODO 14-C
class TinyImageCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = torch.nn.Sequential(
            nn.Conv2d(in_channels = 3, out_channels = 16, kernel_size = 3, padding = 1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(in_channels = 16, out_channels = 32, kernel_size = 3, padding = 1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        with torch.no_grad() : 
            dummy = torch.randn(1,3,48,48)
            output = self.features(dummy)
            num_features = output.flatten(start_dim = 1).shape[1]
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features = num_features, out_features = num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x,start_dim = 1)
        return self.classifier(x)

def make_image_baseline_model(num_classes):
    model = TinyImageCNN(num_classes)
    return model

def count_trainable_parameters(model):
    return int(sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))


# Smoke check: run this after implementing the functions above.
model = make_image_baseline_model(len(label_to_idx)).to(DEVICE)
smoke_logits = model(smoke_batch["image"].to(DEVICE))
print(
    "model smoke check:",
    smoke_logits.shape,
    "trainable parameters:", count_trainable_parameters(model),
)


model smoke check: torch.Size([8, 3]) trainable parameters: 60483


## Exercise 14-D: Training and Validation

Write the training and validation functions.

`train_one_epoch(...)` should train the model and return `loss` and `accuracy`.

`evaluate_model(...)` should run without gradients and return `loss`, `accuracy`, `macro_f1`, `true_indices`, and `pred_indices`.

`macro_f1_from_indices(...)` should compute per-class F1 scores manually and average them.

In [9]:
# TODO 14-D
def macro_f1_from_indices(y_true, y_pred, num_classes):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    F1 = 0
    for i in range(num_classes) :
        TP = ((y_true == y_pred) & (y_true == i)).sum()
        FP = ((y_true != y_pred) & (y_pred == i)).sum()
        FN = ((y_true == i) & (y_pred != i)).sum()
        precision = TP/(TP + FP) if TP + FP > 0 else 0.0
        recall = TP/(TP + FN) if TP + FN > 0 else 0.0
        if precision + recall > 0 :
            F1 += 2 * precision * recall / (precision + recall)
    return F1 / num_classes

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    samples = 0
    total_loss = 0.0
    correct = 0

    for batch in loader : 
        image = batch["image"].to(device)
        label = batch["label"].to(device)
        batch_size = len(image)
        optimizer.zero_grad()
        logits = model(image)
        loss = criterion(logits,label)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_size
        preds = logits.argmax(dim = 1)
        correct += (preds == label).sum().item()
        samples += batch_size
    
    avg_loss = total_loss/samples
    accuracy = correct/samples
    return {
        "loss" : avg_loss,
        "accuracy" : accuracy
    }


def evaluate_model(model, loader, criterion, device):
    model.eval()

    samples = 0
    total_loss = 0.0
    correct = 0
    all_preds = []
    all_labels = []

    with torch.no_grad() : 
        for batch in loader : 
            image = batch["image"].to(device)
            label = batch["label"].to(device)
            batch_size = len(image)
            logits = model(image)
            loss = criterion(logits,label)

            total_loss += loss.item() * batch_size
            preds = logits.argmax(dim = 1)
            correct += (preds == label).sum().item()
            samples += batch_size
            all_preds += preds.tolist()
            all_labels += label.tolist()
    f1 = macro_f1_from_indices(all_labels, all_preds, len(label_to_idx))
    avg_loss = total_loss/samples
    accuracy = correct/samples
    return {
        "loss" : avg_loss,
        "accuracy" : accuracy,
        "macro_f1" : f1,
        "true_indices" : all_labels,
        "pred_indices" : all_preds
    }


# Smoke check: run this after implementing the functions above.
smoke_cpu_rng_state = torch.get_rng_state()
smoke_cuda_rng_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
smoke_f1 = macro_f1_from_indices(
    np.array([0, 1, 2, 2]), np.array([0, 1, 1, 2]), num_classes=3
)
smoke_criterion = nn.CrossEntropyLoss()
smoke_optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
smoke_train_metrics = train_one_epoch(
    model, train_loader, smoke_criterion, smoke_optimizer, DEVICE
)
smoke_val_metrics = evaluate_model(model, val_loader, smoke_criterion, DEVICE)
torch.set_rng_state(smoke_cpu_rng_state)
if smoke_cuda_rng_state is not None:
    torch.cuda.set_rng_state_all(smoke_cuda_rng_state)
print("metric smoke check:", smoke_f1)
print("training smoke check:", smoke_train_metrics)
print("evaluation smoke check:", smoke_val_metrics)


metric smoke check: 0.7777777777777777
training smoke check: {'loss': 0.0017619602631131987, 'accuracy': 1.0}
evaluation smoke check: {'loss': 2.4181257088979087, 'accuracy': 0.3333333333333333, 'macro_f1': np.float64(0.16666666666666666), 'true_indices': [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2], 'pred_indices': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


## Exercise 14-E: Submission and Timed Experiment Loop

Write the final contest-shaped pieces.

`make_submission(...)` should run the model on the test loader, map predicted class indices back to label strings, write `submission.csv`, and return the submission `DataFrame`.

`run_baseline_experiment(...)` should connect the full flow: load records, create loaders, create model, train for a few epochs, evaluate each epoch, and write a submission file.

In [7]:
# TODO 14-E
def make_submission(model, test_loader, idx_to_label, output_csv=SUBMISSION_CSV, device=DEVICE):
    preds = []
    image_ids = []
    with torch.no_grad() : 
        for batch in test_loader : 
            image = batch["image"].to(device)
            logits = model(image)
            image_id = batch["image_id"]
            pred = logits.argmax(dim = 1)
            preds += pred.tolist()
            image_ids += image_id
    preds = [idx_to_label[i] for i in preds]
    submission = pd.DataFrame({
        "image_id" : image_ids,
        "label" : preds
    })
    submission.to_csv(output_csv,index = False)
    return submission


def run_baseline_experiment(epochs=2, batch_size=8, lr=0.003, device=DEVICE):
    train_records, val_records, test_records, label_to_idx, idx_to_label = load_baseline_records()
    train_loader, val_loader, test_loader = make_baseline_loaders(
        train_records,
        val_records,
        test_records,
        label_to_idx,
        batch_size=batch_size,
    )
    model = make_image_baseline_model(len(label_to_idx)).to(device)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr = lr)
    history = []
    for epoch in range(epochs):
        train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics = evaluate_model(model, val_loader, criterion, device)
        history.append({
            "epoch": epoch + 1,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
        })

    submission = make_submission(model, test_loader, idx_to_label, SUBMISSION_CSV, device)
    return model, pd.DataFrame(history), submission




# Smoke check: run this after implementing the functions above.
smoke_model, smoke_history, smoke_submission = run_baseline_experiment(epochs=1)
print("experiment smoke check:")
print(smoke_history)
print(smoke_submission.head())


experiment smoke check:
   epoch  train_loss  train_accuracy  val_loss  val_accuracy  val_macro_f1
0      1    1.767827        0.694444  0.843643      0.333333      0.166667
           image_id   label
0    test_circle_00  square
1    test_circle_01  square
2    test_square_00  square
3    test_square_01  square
4  test_triangle_00  square


## Test Cases

Run this cell after completing the TODO cells above. A correct implementation should print `Day 14 tests passed`.

In [10]:
def run_day14_tests():
    assert os.path.isdir(IMAGE_DIR), "Missing image directory"
    assert os.path.exists(LABELS_CSV), "Missing labels.csv"
    assert os.path.exists(TEST_CSV), "Missing test.csv"

    required_names = [
        "load_baseline_records",
        "image_to_tensor",
        "BaselineImageDataset",
        "make_baseline_loaders",
        "TinyImageCNN",
        "make_image_baseline_model",
        "count_trainable_parameters",
        "macro_f1_from_indices",
        "train_one_epoch",
        "evaluate_model",
        "make_submission",
        "run_baseline_experiment",
    ]
    for name in required_names:
        assert name in globals(), "Missing function or class: " + name

    train_records, val_records, test_records, label_to_idx, idx_to_label = load_baseline_records()
    assert len(label_to_idx) == 3
    assert set(label_to_idx.keys()) == set(LABELS)
    assert set(idx_to_label.values()) == set(LABELS)
    assert len(train_records) == 36
    assert len(val_records) == 12
    assert len(test_records) == 6
    assert set(train_records["split"]) == {"train"}
    assert set(val_records["split"]) == {"val"}

    first_path = os.path.join(DATA_DIR, train_records.iloc[0]["image_path"])
    sample_tensor = image_to_tensor(Image.open(first_path))
    assert tuple(sample_tensor.shape) == (3, IMAGE_SIZE, IMAGE_SIZE)
    assert sample_tensor.dtype == torch.float32
    assert float(sample_tensor.abs().mean()) > 0.05

    train_loader, val_loader, test_loader = make_baseline_loaders(
        train_records,
        val_records,
        test_records,
        label_to_idx,
        batch_size=5,
    )
    batch = next(iter(train_loader))
    assert tuple(batch["image"].shape[1:]) == (3, IMAGE_SIZE, IMAGE_SIZE)
    assert batch["label"].dtype == torch.long
    assert len(batch["image_id"]) == batch["image"].shape[0]

    cpu_device = torch.device("cpu")
    model = make_image_baseline_model(len(label_to_idx)).to(cpu_device)
    logits = model(batch["image"].to(cpu_device))
    assert tuple(logits.shape) == (batch["image"].shape[0], len(label_to_idx))
    assert count_trainable_parameters(model) > 1000

    f1 = macro_f1_from_indices([0, 0, 1, 1, 2], [0, 1, 1, 1, 2], 3)
    assert 0.0 <= f1 <= 1.0
    assert np.isclose(f1, (0.6666666667 + 0.8 + 1.0) / 3.0)

    criterion = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=0.001)
    train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, cpu_device)
    assert set(train_metrics.keys()) == {"loss", "accuracy"}
    assert np.isfinite(train_metrics["loss"])
    assert 0.0 <= train_metrics["accuracy"] <= 1.0

    val_metrics = evaluate_model(model, val_loader, criterion, cpu_device)
    assert {"loss", "accuracy", "macro_f1", "true_indices", "pred_indices"}.issubset(val_metrics.keys())
    assert len(val_metrics["true_indices"]) == len(val_records)
    assert len(val_metrics["pred_indices"]) == len(val_records)
    assert 0.0 <= val_metrics["macro_f1"] <= 1.0

    submission = make_submission(model, test_loader, idx_to_label, output_csv=SUBMISSION_CSV, device=cpu_device)
    assert list(submission.columns) == ["image_id", "label"]
    assert len(submission) == len(test_records)
    assert set(submission["label"]).issubset(set(LABELS))
    assert os.path.exists(SUBMISSION_CSV)

    print("Day 14 tests passed")


run_day14_tests()


Day 14 tests passed


## Day 14 Checklist

- Create an image dataset table with train, validation, and test records.
- Build one stable label mapping and reuse it everywhere.
- Confirm batch image shape, dtype, device, and label dtype before training.
- Train a small baseline model and track validation loss, accuracy, and Macro-F1.
- Convert predicted indices back to label names and write a submission-style CSV.
- Practice the whole flow as a timed baseline from a blank notebook.